# 01 — Data Cleaning
> **Superstore Sales Analytics | Future Interns DS Task 1**

This notebook handles all raw data ingestion, quality checks, and cleaning steps.  
Output: `dataset/processed/superstore_clean.csv` — the single source of truth used by all downstream notebooks.

---
### Steps
1. Load raw CSV (handle encoding)
2. Inspect shape, dtypes, and sample rows
3. Check for missing values
4. Check for duplicates
5. Fix data types (dates, numerics)
6. Engineer new columns (Year, Month, Quarter, Profit Margin, Shipping Days)
7. Validate and export clean data

## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

RAW_PATH = '../dataset/raw/Superstore.csv'
CLEAN_PATH = '../dataset/processed/superstore_clean.csv'

os.makedirs('../dataset/processed', exist_ok=True)

print('Libraries loaded ✓')

Libraries loaded ✓


## 2. Load Raw Data

In [2]:
# The file uses latin-1 encoding — utf-8 will raise a UnicodeDecodeError
df = pd.read_csv(RAW_PATH, encoding='latin-1')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

Shape: 9,994 rows × 21 columns


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


In [3]:
# Column names and dtypes at a glance
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

## 3. Missing Value Audit

In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0')

if missing_report.empty:
    print('✅ No missing values found across all 21 columns.')
else:
    print(missing_report)

✅ No missing values found across all 21 columns.


## 4. Duplicate Check

In [5]:
# Row-level duplicates (all 21 columns identical)
full_dupes = df.duplicated().sum()
print(f'Full-row duplicates: {full_dupes}')

# Order ID appears multiple times — this is EXPECTED
# One order can have many line items (products)
order_id_dupes = df['Order ID'].duplicated().sum()
unique_orders = df['Order ID'].nunique()
print(f'\nUnique orders (Order IDs): {unique_orders:,}')
print(f'Total rows: {len(df):,}')
print(f'→ Average {len(df)/unique_orders:.1f} line items per order (expected, not a data error)')

if full_dupes > 0:
    df = df.drop_duplicates()
    print(f'\nDropped {full_dupes} exact duplicate rows. New shape: {df.shape}')

Full-row duplicates: 0

Unique orders (Order IDs): 5,009
Total rows: 9,994
→ Average 2.0 line items per order (expected, not a data error)


## 5. Fix Data Types

In [6]:
# Parse date columns (stored as strings in M/D/YYYY format)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

# Postal Code should be a string (leading zeros matter for some states)
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)

# Confirm numeric columns are correct dtype
for col in ['Sales', 'Quantity', 'Discount', 'Profit']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Dtypes after fixing:')
print(df[['Order Date', 'Ship Date', 'Postal Code', 'Sales', 'Profit']].dtypes)

Dtypes after fixing:
Order Date     datetime64[us]
Ship Date      datetime64[us]
Postal Code               str
Sales                 float64
Profit                float64
dtype: object


## 6. Feature Engineering

In [7]:
# Time-based features — used heavily in EDA and dashboard
df['Year']          = df['Order Date'].dt.year
df['Month']         = df['Order Date'].dt.month
df['Month Name']    = df['Order Date'].dt.strftime('%b')   # Jan, Feb …
df['Quarter']       = df['Order Date'].dt.quarter
df['Quarter Label'] = 'Q' + df['Quarter'].astype(str)
df['YearMonth']     = df['Order Date'].dt.to_period('M')   # 2014-01, 2014-02 …

# Shipping speed (days between order and delivery)
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Profit margin per row (avoid division by zero)
df['Profit Margin'] = np.where(
    df['Sales'] != 0,
    df['Profit'] / df['Sales'],
    0
)

# Discount band — useful for binned analysis
df['Discount Band'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.2, 0.4, 1.0],
    labels=['No Discount', 'Low (1–20%)', 'Medium (21–40%)', 'High (41%+)']
)

print(f'New columns added: {["Year","Month","Month Name","Quarter","Quarter Label","YearMonth","Shipping Days","Profit Margin","Discount Band"]}')
df[['Order Date', 'Year', 'Quarter Label', 'Shipping Days', 'Profit Margin', 'Discount Band']].head()

New columns added: ['Year', 'Month', 'Month Name', 'Quarter', 'Quarter Label', 'YearMonth', 'Shipping Days', 'Profit Margin', 'Discount Band']


,Order Date,Year,Quarter Label,Shipping Days,Profit Margin,Discount Band
0,2016-11-08,2016,Q4,3,0.16,No Discount
1,2016-11-08,2016,Q4,3,0.30,No Discount
2,2016-06-12,2016,Q2,4,0.47,No Discount
3,2015-10-11,2015,Q4,7,-0.40,High (41%+)
4,2015-10-11,2015,Q4,7,0.11,Low (1–20%)


## 7. Quick Sanity Checks

In [8]:
# Negative profits are real (heavy discounts) — flag but don't remove
negative_profit = (df['Profit'] < 0).sum()
print(f'Rows with negative profit: {negative_profit:,} ({negative_profit/len(df)*100:.1f}%)')

# Shipping days shouldn't be negative
bad_ship = (df['Shipping Days'] < 0).sum()
print(f'Rows with negative shipping days: {bad_ship}')

# Year coverage
print(f'\nYears covered: {sorted(df["Year"].unique())}')

# Value ranges
print(f'\nSales range:   ${df["Sales"].min():.2f} — ${df["Sales"].max():,.2f}')
print(f'Profit range:  ${df["Profit"].min():,.2f} — ${df["Profit"].max():,.2f}')
print(f'Discount range: {df["Discount"].min()} — {df["Discount"].max()}')
print(f'Shipping Days range: {df["Shipping Days"].min()} — {df["Shipping Days"].max()}')

Rows with negative profit: 1,871 (18.7%)
Rows with negative shipping days: 0

Years covered: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017)]

Sales range:   $0.44 — $22,638.48
Profit range:  $-6,599.98 — $8,399.98
Discount range: 0.0 — 0.8
Shipping Days range: 0 — 7


In [9]:
# Categorical value counts — verify no typos or unexpected values
for col in ['Category', 'Region', 'Segment', 'Ship Mode']:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())


Category:
Category
Office Supplies    6026
Furniture          2121
Technology         1847

Region:
Region
West       3203
East       2848
Central    2323
South      1620

Segment:
Segment
Consumer       5191
Corporate      3020
Home Office    1783

Ship Mode:
Ship Mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543


## 8. Final Summary & Export

In [10]:
print('=' * 50)
print('CLEAN DATASET SUMMARY')
print('=' * 50)
print(f'Rows             : {len(df):,}')
print(f'Columns          : {len(df.columns)}')
print(f'Date range       : {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Total Revenue    : ${df["Sales"].sum():,.2f}')
print(f'Total Profit     : ${df["Profit"].sum():,.2f}')
print(f'Overall Margin   : {df["Profit"].sum() / df["Sales"].sum() * 100:.1f}%')
print(f'Unique Customers : {df["Customer ID"].nunique():,}')
print(f'Unique Products  : {df["Product ID"].nunique():,}')
print(f'Missing Values   : {df.isnull().sum().sum()}')
print('=' * 50)

# Export — exclude the Period column (not CSV-serialisable)
df.drop(columns=['YearMonth']).to_csv(CLEAN_PATH, index=False)
print(f'\n✅ Clean data saved → {CLEAN_PATH}')

CLEAN DATASET SUMMARY
Rows             : 9,994
Columns          : 30
Date range       : 2014-01-03 → 2017-12-30
Total Revenue    : $2,297,200.86
Total Profit     : $286,397.02
Overall Margin   : 12.5%
Unique Customers : 793
Unique Products  : 1,862
Missing Values   : 0

✅ Clean data saved → ../dataset/processed/superstore_clean.csv
